## Weather API

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

API_KEY = os.getenv('AZURE_OPENAI_API_KEY')
BASE_URL = os.getenv('OPENAI_BASE_URL')

WEATHER_KEY = os.getenv('WEATHER_KEY')

os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ["LANGCHAIN_TRACING_V2"] = "true" 
os.environ['LANGSMITH_PROJECT'] = 'AgenticAITraining' 

In [2]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model = 'gpt-5.1',
    api_key = API_KEY,
    base_url = BASE_URL
)

In [3]:
import requests
from langchain.tools import tool

c:\Data\Codes\AgenticAITraining\Lanchain_Azure_openAI\.venv\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [4]:
@tool
def get_weather_data(city: str) -> str:
    '''This function fetches the current weather data for a given city, with country'''
    url = f'https://api.weatherstack.com/current?access_key={WEATHER_KEY}&query={city}'
    response = requests.get(url)

    return response.json()


In [5]:
get_weather_data.invoke({'city': 'New Delhi'})

{'request': {'type': 'City',
  'query': 'New Delhi, India',
  'language': 'en',
  'unit': 'm'},
 'location': {'name': 'New Delhi',
  'country': 'India',
  'region': 'Delhi',
  'lat': '28.600',
  'lon': '77.200',
  'timezone_id': 'Asia/Kolkata',
  'localtime': '2026-05-14 16:56',
  'localtime_epoch': 1778777760,
  'utc_offset': '5.50'},
 'current': {'observation_time': '11:26 AM',
  'temperature': 39,
  'weather_code': 143,
  'weather_icons': ['https://cdn.worldweatheronline.com/images/wsymbols01_png_64/wsymbol_0006_mist.png'],
  'weather_descriptions': ['Haze'],
  'astro': {'sunrise': '05:32 AM',
   'sunset': '07:04 PM',
   'moonrise': '03:20 AM',
   'moonset': '04:27 PM',
   'moon_phase': 'Waning Crescent',
   'moon_illumination': 11},
  'air_quality': {'co': '484.85',
   'no2': '18.65',
   'o3': '241',
   'so2': '52.95',
   'pm2_5': '97.25',
   'pm10': '346.35',
   'us-epa-index': '4',
   'gb-defra-index': '4'},
  'wind_speed': 22,
  'wind_degree': 280,
  'wind_dir': 'W',
  'pressure

In [6]:
from langchain.agents import create_agent
tools = [get_weather_data]  # There can be multiple tools out here

agent = create_agent(
    model = llm,
    tools = tools,
    system_prompt="Answer the following questions as best you can. You have access to the following tools",
)

In [7]:
# Invoking the agent

response = agent.invoke(
    {'messages': 'What is the current temprature at the Capital City of Andaman and Nicobar, India'}
)

print(response['messages'][-1].content)

The current temperature in Port Blair, the capital of Andaman and Nicobar Islands, India, is **28°C**.


In [8]:
response = agent.invoke(
    {'messages': 'What is the current temprature at Gurugram and South Delhi'}
)

print(response['messages'][-1].content)

I’m currently rate-limited from fetching *both* locations live, but I do have fresh data for Delhi (which is representative of South Delhi).

- **South Delhi (Delhi, India)**  
  - Current temperature: **39°C**  
  - Conditions: **Hazy** (reported as “Haze”)  
  - Feels-like: **38°C** (low humidity but hot)  

- **Gurugram**  
  I couldn’t retrieve this in parallel due to a rate limit, but typically Gurugram’s temperature is within about **±1–2°C** of South Delhi’s at the same time. So you can reasonably assume it is currently around **37–41°C**, very similar to South Delhi’s conditions.

For exact, up-to-the-minute values for Gurugram, you may want to quickly check a weather app (e.g., IMD, Weather.com, or AccuWeather) until I can make another live call.


In [9]:
for msg in response['messages']:
    msg.pretty_print()

================================ Human Message =================================

What is the current temprature at Gurugram and South Delhi
================================== Ai Message ==================================
Tool Calls:
  get_weather_data (call_I6nULWoOLlh3NIbuAxJBNwL6)
 Call ID: call_I6nULWoOLlh3NIbuAxJBNwL6
  Args:
    city: Gurugram
  get_weather_data (call_G69LsZ7rLuJaeAVYfB9rQe0A)
 Call ID: call_G69LsZ7rLuJaeAVYfB9rQe0A
  Args:
    city: South Delhi
================================= Tool Message =================================
Name: get_weather_data

{"success": false, "error": {"code": 106, "type": "rate_limit_reached", "info": "You have exceeded the maximum rate limitation allowed on your subscription plan. Please refer to the \"Rate Limits\" section of the API Documentation for details. "}}
================================= Tool Message =================================
Name: get_weather_data

{"request": {"type": "City", "query": "Delhi, India", "language": "en